# Project 13: Heart Disease Risk Prediction
**Team No.:** 19  
**Team Members:** Shubham Kumar Pradhan; Srabani Mohanty; Uttam Biswal; Satyajeet Mohapatra  
**Proposed Hybrid Model:** FT-Transformer + Residual Tabular MLP  
**Dataset:** Cardiovascular disease dataset (cardio_train.csv)  
**Source:** https://www.kaggle.com/datasets/sulianova/cardiovascular-disease-dataset


## 0. Setup — Environment, Imports, Reproducibility


In [ ]:
!pip -q install kaggle tqdm tabulate
import os, json, random, glob, subprocess, sys, math, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, average_precision_score, confusion_matrix, mean_absolute_error, mean_squared_error, r2_score
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
assert torch.cuda.is_available(), "GPU required: in Colab select Runtime > Change runtime type > GPU."
DEVICE = torch.device("cuda:0")
print("Using device:", DEVICE)


### CONFIG


In [ ]:
CONFIG = {
 "project_no":"13", "project_name":"Heart Disease Risk Prediction", "team_no":"19",
 "task_type":"classification", "kaggle_dataset_slug":"sulianova/cardiovascular-disease-dataset",
 "target_column":None, "id_columns":[], "time_column":None,
 "split_ratios":{"train":0.70,"val":0.15,"test":0.15}, "random_seed":SEED,
 "data_raw_dir":"data/13/raw", "data_processed_dir":"data/13/processed", "figures_dir":"data/13/figures", "results_dir":"data/13/results",
 "reports_dir":"data/13/reports", "batch_size":64, "epochs":30, "patience":5
}
for key in ["data_raw_dir","data_processed_dir","figures_dir","results_dir","reports_dir"]: os.makedirs(CONFIG[key],exist_ok=True)
CONFIG


## 1. Dataset Download


In [ ]:
raw=Path(CONFIG["data_raw_dir"])
if not any(raw.rglob("*")):
    subprocess.run(["kaggle","datasets","download","-d",CONFIG["kaggle_dataset_slug"],"-p",str(raw),"--unzip"],check=True)
for z in raw.rglob("*.zip"):
    import zipfile
    with zipfile.ZipFile(z) as f: f.extractall(z.parent/z.stem)
raw_files=[p for p in raw.rglob("*") if p.is_file()]
assert raw_files, "Dataset download produced no files. Configure Kaggle credentials in Colab and rerun."
assert sum(p.stat().st_size for p in raw_files)>1024, "Downloaded content is unexpectedly small."
print(f"Discovered {len(raw_files)} files; {sum(p.stat().st_size for p in raw_files)/2**20:.1f} MiB")


## 2. Load Raw Data


In [ ]:
csvs=list(Path(CONFIG["data_raw_dir"]).rglob("*.csv")); assert csvs,"No CSV found"
if CONFIG["project_no"]=="13":
    path=max(csvs,key=lambda p:p.stat().st_size); df=pd.read_csv(path,sep=None,engine="python")
    aliases=["cardio","target","label"]
else:
    candidates=[p for p in csvs if "dataset" in p.name.lower() or "training" in p.name.lower()]
    path=max(candidates or csvs,key=lambda p:p.stat().st_size); df=pd.read_csv(path)
    aliases=["prognosis","disease","label","target"]
target=next((c for a in aliases for c in df.columns if c.strip().lower()==a),None)
assert target is not None, f"Target not found. Columns: {df.columns.tolist()}"
CONFIG["target_column"]=target; df=df.drop_duplicates(); df=df[df[target].notna()].reset_index(drop=True)
print(path,df.shape,target); display(df.head())


## 3. Exploratory Data Analysis (EDA) + Data Quality Memo


In [ ]:
print(df.info()); missing=df.isna().mean().sort_values(ascending=False)
memo=f"""# Data quality memo
- Rows: {len(df):,}; columns: {df.shape[1]}.
- Duplicate rows were removed before splitting.
- Maximum missing fraction: {missing.max():.3f}.
- Preprocessors are fit only on training data.
- Stratification preserves outcome prevalence; the held-out test set is evaluated once.
"""
Path("reports/data_quality_memo.md").write_text(memo,encoding="utf-8")


## 4. Preprocessing & Feature Engineering


Feature construction is performed after splitting; every learned imputer, scaler, encoder, graph, and vocabulary is fit on training data only.


## 5. Train / Validation / Test Split


In [ ]:
train_df,rest=train_test_split(df,test_size=.30,stratify=df[target],random_state=SEED)
val_df,test_df=train_test_split(rest,test_size=.50,stratify=rest[target],random_state=SEED)
assert not (set(train_df.index)&set(test_df.index)); assert not (set(val_df.index)&set(test_df.index))
drop_cols=[target]+[c for c in df.columns if c.lower() in {"id","patient_id"}]
feature_cols=[c for c in df.columns if c not in drop_cols]
for c in feature_cols:
    if not pd.api.types.is_numeric_dtype(train_df[c]):
        vocab={v:i+1 for i,v in enumerate(train_df[c].astype(str).unique())}
        for part in (train_df,val_df,test_df): part.loc[:,c]=part[c].astype(str).map(vocab).fillna(0)
imputer=SimpleImputer(strategy="median").fit(train_df[feature_cols]); scaler=StandardScaler().fit(imputer.transform(train_df[feature_cols]))
le=LabelEncoder().fit(train_df[target].astype(str))
def transform(part): return scaler.transform(imputer.transform(part[feature_cols])).astype("float32"),le.transform(part[target].astype(str)).astype("int64")
Xtr,ytr=transform(train_df); Xv,yv=transform(val_df); Xte,yte=transform(test_df)
manifest={"train_rows":len(ytr),"val_rows":len(yv),"test_rows":len(yte),"classes":le.classes_.tolist()}
Path("data/processed/split_manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")


## 6. PyTorch Dataset & DataLoader


In [ ]:
class TabularDataset(Dataset):
    def __init__(self,x,y): self.x=torch.tensor(x); self.y=torch.tensor(y)
    def __len__(self): return len(self.y)
    def __getitem__(self,i): return self.x[i],self.y[i]
train_loader=DataLoader(TabularDataset(Xtr,ytr),batch_size=CONFIG["batch_size"],shuffle=True)
val_loader=DataLoader(TabularDataset(Xv,yv),batch_size=CONFIG["batch_size"])
test_loader=DataLoader(TabularDataset(Xte,yte),batch_size=CONFIG["batch_size"])


## 7. Proposed Model Definition


In [ ]:
class FeatureTokenizer(nn.Module):

    def __init__(self, d, h):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(d, h) * 0.02)
        self.bias = nn.Parameter(torch.zeros(d, h))

    def forward(self, x):
        return x.unsqueeze(-1) * self.weight + self.bias

class FTTransformerResidualMLP(nn.Module):

    def __init__(self, d, k, h=64):
        super().__init__()
        self.token = FeatureTokenizer(d, h)
        layer = nn.TransformerEncoderLayer(h, 4, 128, batch_first=True, dropout=0.1)
        self.ft = nn.TransformerEncoder(layer, 2)
        self.residual = nn.Sequential(nn.Linear(d, 128), nn.ReLU(), nn.Linear(128, h))
        self.head = nn.Sequential(nn.LayerNorm(2 * h), nn.Linear(2 * h, k))

    def forward(self, x):
        return self.head(torch.cat([self.ft(self.token(x)).mean(1), self.residual(x)], 1))


## 8. Training Loop


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    model.train(optimizer is not None)
    total = 0.0
    n = 0
    for xb, yb in loader:
        xb, yb = (xb.to(DEVICE), yb.to(DEVICE))
        if optimizer:
            optimizer.zero_grad(set_to_none=True)
        out = model(xb)
        loss = criterion(out, yb)
        if optimizer:
            loss.backward()
            optimizer.step()
        total += loss.item() * len(yb)
        n += len(yb)
    return total / max(n, 1)

def train_model(model, train_loader, val_loader, checkpoint, classification=False):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss() if classification else nn.MSELoss()
    opt = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.0001)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=2, factor=0.5)
    history = {'train_loss': [], 'val_loss': []}
    best = float('inf')
    stale = 0
    for epoch in tqdm(range(CONFIG['epochs']), desc='Training', unit='epoch'):
        tr = run_epoch(model, train_loader, criterion, opt)
        with torch.no_grad():
            va = run_epoch(model, val_loader, criterion)
        history['train_loss'].append(tr)
        history['val_loss'].append(va)
        scheduler.step(va)
        print(f'epoch={epoch + 1:02d} train={tr:.5f} val={va:.5f}')
        if va < best:
            best = va
            stale = 0
            torch.save(model.state_dict(), checkpoint)
        else:
            stale += 1
            if stale >= CONFIG['patience']:
                break
    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE, weights_only=True))
    return (model, history)

def predict(model, loader, classification=False):
    model.eval()
    pred = []
    true = []
    with torch.no_grad():
        for xb, yb in loader:
            out = model(xb.to(DEVICE)).cpu()
            pred.append(torch.softmax(out, 1) if classification else out)
            true.append(yb)
    return (torch.cat(pred).numpy(), torch.cat(true).numpy())
hybrid = FTTransformerResidualMLP(Xtr.shape[1], len(le.classes_))
hybrid, hybrid_history = train_model(hybrid, train_loader, val_loader, 'results/best_hybrid.pt', True)


## 9. Evaluation Metrics


In [ ]:
results = {}
cached = {}
for name, model in [('hybrid', hybrid)]:
    probs, y = predict(model, test_loader, True)
    pred = probs.argmax(1)
    cached[name] = (probs, pred, y)
    pr, re, f1, _ = precision_recall_fscore_support(y, pred, average='macro', zero_division=0)
    results[name] = {'accuracy': accuracy_score(y, pred), 'precision_macro': pr, 'recall_macro': re, 'f1_macro': f1}
    if probs.shape[1] == 2:
        results[name]['roc_auc'] = roc_auc_score(y, probs[:, 1])
        results[name]['pr_auc'] = average_precision_score(y, probs[:, 1])
Path('results/metrics.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
results


## 10. Required Figures


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(hybrid_history['train_loss'], label='hybrid train')
plt.plot(hybrid_history['val_loss'], label='hybrid val')
plt.legend()
plt.tight_layout()
plt.savefig('figures/fig01_loss_curves.png', dpi=150)
plt.show()
if CONFIG['task_type'] == 'classification':
    probs, pred, y = cached['hybrid']
    cm = confusion_matrix(y, pred)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig('figures/fig02_confusion_matrix.png', dpi=150)
    plt.show()
    support = np.bincount(y, minlength=probs.shape[1])
    per_class = [(pred[y == i] == i).mean() if (y == i).any() else np.nan for i in range(probs.shape[1])]
    plt.figure(figsize=(9, 4))
    plt.bar(range(len(per_class)), per_class)
    plt.ylabel('Per-class recall')
    plt.tight_layout()
    plt.savefig('figures/fig03_class_performance.png', dpi=150)
    plt.show()
else:
    pred, y = cached['hybrid']
    plt.figure(figsize=(6, 5))
    plt.scatter(y.ravel(), pred.ravel(), s=8, alpha=0.35)
    lo = min(y.min(), pred.min())
    hi = max(y.max(), pred.max())
    plt.plot([lo, hi], [lo, hi], 'k--')
    plt.xlabel('Actual')
    plt.ylabel('Predicted')
    plt.tight_layout()
    plt.savefig('figures/fig02_predicted_vs_actual.png', dpi=150)
    plt.show()
    residual = (pred - y).ravel()
    plt.figure(figsize=(7, 4))
    sns.histplot(residual, bins=40, kde=True)
    plt.xlabel('Residual')
    plt.tight_layout()
    plt.savefig('figures/fig03_residual_distribution.png', dpi=150)
    plt.show()
xb, yb = next(iter(test_loader))
xb = xb[:min(32, len(xb))].to(DEVICE).requires_grad_(True)
hybrid.zero_grad()
out = hybrid(xb)
score = out.max(1).values.sum() if out.ndim == 2 and out.shape[1] > 1 else out.sum()
score.backward()
importance = xb.grad.detach().abs().cpu().numpy()
imp = importance.mean(axis=tuple(range(importance.ndim - 1))) if importance.ndim > 2 else importance.mean(0)
plt.figure(figsize=(8, 4))
plt.plot(np.ravel(imp))
plt.title('Gradient-based input importance')
plt.tight_layout()
plt.savefig('figures/fig04_feature_importance.png', dpi=150)
plt.show()
if CONFIG['task_type'] == 'classification':
    errors = (cached['hybrid'][1] != cached['hybrid'][2]).astype(int)
else:
    errors = np.abs(cached['hybrid'][0] - cached['hybrid'][1]).reshape(len(cached['hybrid'][1]), -1).mean(1)
plt.figure(figsize=(8, 4))
plt.hist(errors, bins=30)
plt.title('Held-out error distribution')
plt.tight_layout()
plt.savefig('figures/fig05_error_analysis.png', dpi=150)
plt.show()
metric = next(iter(results['hybrid']))
plt.figure(figsize=(6, 4))
plt.bar(results.keys(), [results[k][metric] for k in results])
plt.ylabel(metric)
plt.tight_layout()
plt.savefig('figures/fig06_proposed_metrics.png', dpi=150)
plt.show()
